> **참고용 사본 (reference copy).** 원본 노트북에서 출력, 실행 횟수, 로컬 경로, 키 값을 제거한 사본입니다. 비공개 원천 데이터가 필요해 이 저장소만으로는 실행되지 않으며, 코드 구조와 분석 절차를 보여주는 용도입니다.

*Reference copy of a private notebook: outputs and paths removed; requires non-public data to run.*


<!-- report_markdown_v2 -->
# 배당 검색 관심도와 주식수익률 실증분석 보고

본 노트북은 배당 관련 검색량 변화가 한국 주식의 횡단면 수익률을 설명하는지 검증한다.  
검색 관심도 변수는 네이버 데이터랩의 `배당주` 검색량 변화율로 구성한다.

분석 절차는 다음과 같다.

1. 종목별 배당 관심도 베타(`beta_divatt`)를 월별로 추정한다.
2. `beta_mkt`와 `beta_divatt` 기준으로 5x5 base portfolio를 구성한다.
3. 25개 포트폴리오를 test asset으로 사용해 2-step regression을 수행한다.
4. `DIVATT` 요인의 위험가격과 분위 포트폴리오 알파를 확인한다.


<!-- literature_gap_v1 -->
## 1. 연구의 위치와 차별성

가장 직접적인 벤치마크는 Kumar, Lei, and Zhang(2022)의 배당 검색심리 연구이다.  
이 연구는 Google 검색량으로 투자자의 배당 선호를 측정하고, 배당 검색심리가 고배당주 수요와 단기 수익률 예측력으로 이어질 수 있음을 보였다.

국외 연구는 크게 세 흐름으로 진행되어 왔다.  
첫째, 배당수익률이나 배당프리미엄처럼 시장가격에 반영된 배당지표가 수익률을 예측하는지 검증하였다.  
둘째, 배당 케이터링 이론은 투자자의 배당 선호가 기업의 배당정책에 영향을 주는지 분석하였다.  
셋째, 최근 연구는 검색량을 이용해 배당 선호를 더 직접적으로 측정하고, 고배당주 수요와 수익률 반응을 검증하였다.

국내 배당 관련 연구도 존재하지만, 대부분은 다음 범위에 머물러 있다.  
첫째, 배당수익률과 주식수익률의 예측 관계를 분석한 연구이다.  
둘째, 배당공시와 배당락일 전후의 시장반응을 검증한 연구이다.  
셋째, 외국인 또는 투자자 심리와 배당정책의 관계를 분석한 연구이다.  
넷째, 네이버 검색량과 주식수익률을 연결한 연구는 있으나, 일반적인 기업 검색량을 투자자 관심 변수로 사용하였다.

따라서 확인한 국내 문헌 기준으로는 `배당 관련 검색량`을 이용해 투자자의 배당 관심도를 직접 측정하고, 이를 개별 종목의 민감도 베타와 5x5 포트폴리오, 2-step regression으로 자산가격결정 관점에서 검증한 연구는 찾기 어렵다.

본 연구의 기여는 이 공백을 메우는 데 있다.  
즉, 배당을 기업의 재무정책이나 배당수익률 지표로만 보지 않고, 투자자가 배당을 얼마나 적극적으로 탐색하는지라는 수요 측 관심 변수로 확장한다.

한국시장은 이 질문을 검증하기에 적합하다.  
개인투자자의 거래 영향력이 크고, 네이버 검색량이 국내 개인투자자의 정보탐색 행태를 잘 반영할 가능성이 높다.  
또한 KRX는 투자주체별 거래자료를 제공하므로, 향후 배당 검색 관심도 효과가 개인, 기관, 외국인 중 어느 투자자 집단의 거래를 통해 가격에 반영되는지까지 확장할 수 있다.

핵심 참고문헌은 다음과 같다.  
Kumar, Lei, and Zhang(2022)은 배당 검색심리와 수익률 예측력을 직접 분석한 가장 가까운 벤치마크이다.  
유제완·안성진(2025)은 국내 배당수익률과 다음 달 주식수익률의 예측 관계를 분석한 최근 국내 배당 연구이다.  
김류미(2018)는 네이버 검색량이 개인투자자 거래와 일시적 주식수익률 상승에 연결됨을 보인 국내 검색량 연구이다.


## 2. 데이터 및 변수 구성

분석 대상은 금융업을 제외한 한국 상장주식이다.  
수익률은 현금배당 반영 수정주가로 계산하며, 무위험수익률을 차감해 초과수익률을 사용한다.

배당 검색 관심도는 다음과 같이 정의한다.

$$
DIVATT_d = \frac{SVI_d - SVI_{d-1}}{SVI_{d-1}}
$$

현재 분석에서는 `배당주` 키워드의 일별 검색량 변화율을 사용한다.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed

adj_close = pd.read_csv("./input/현금배당반영수정주가_금융업제외.csv", index_col=0, parse_dates=True)
SVI       = pd.read_csv("./input/SVI.csv", index_col=0, parse_dates=True)
MKT       = pd.read_csv("./input/MKT.csv", index_col=0, parse_dates=True).squeeze()
factors   = pd.read_csv("./input/FACTORS.csv", index_col=0, parse_dates=True)

SVI['배당키워드'] = SVI.sum(axis=1)

rf = factors['RF']
rf_daily = (1 + rf / 100) ** (1/252) - 1

In [ ]:
returns = adj_close.pct_change(fill_method=None)

In [ ]:
# SVI_diff = np.log(SVI.reindex(adj_close.index)).diff()
SVI_diff = SVI.reindex(returns.index).pct_change(fill_method=None)
SVI_diff.tail()

In [ ]:
keyword = '배당주'
DIVATT = SVI_diff[keyword]

## 3. Pre-beta 추정

매월 말 $\tau$마다 직전 20거래일 자료로 종목별 베타를 추정한다.  
종목 $i$의 일별 초과수익률 회귀식은 다음과 같다.

$$
r^e_{i,d} = \alpha_{i,\tau}
+ \beta^{MKT}_{i,\tau} MKT^e_d
+ \beta^{DIV}_{i,\tau} DIVATT_d
+ \varepsilon_{i,d}
$$

여기서 $\beta^{DIV}_{i,\tau}$는 배당 검색 관심도 변화에 대한 종목의 민감도이다.


In [ ]:
month_ends  = pd.date_range(start='2016-06-30', end='2025-12-31', freq='ME')
month_ends

In [ ]:
beta_mkt_df = pd.DataFrame(index=month_ends[:-1], columns=returns.columns, dtype=float)
beta_divatt_df = pd.DataFrame(index=month_ends[:-1], columns=returns.columns, dtype=float)

def estimate_one_month(i, month_ends, returns, MKT, DIVATT, rf_daily):
    start_date = month_ends[i]

    ret = returns.loc[:start_date].tail(20)
    ret = ret.sub(rf_daily.reindex(ret.index), axis=0)

    mkt_excess = MKT.reindex(ret.index).sub(rf_daily.reindex(ret.index), axis=0).rename('mkt')
    divatt = DIVATT.reindex(ret.index).rename('divatt')
    X_base = pd.concat([mkt_excess, divatt], axis=1).dropna()

    # mkt_excess = MKT.loc[ret.index].rename('mkt')
    # divatt = DIVATT.loc[ret.index].rename('divatt')
    # X_base = pd.concat([mkt_excess, divatt], axis=1).dropna()

    X = sm.add_constant(X_base)

    ret_aligned = ret.reindex(X_base.index)

    beta_mkt = pd.Series(index=returns.columns, dtype=float)
    beta_divatt = pd.Series(index=returns.columns, dtype=float)

    complete_cols = ret_aligned.columns[ret_aligned.notna().all(axis=0)]

    for col in complete_cols:
        y = ret_aligned[col]
        res = sm.OLS(y, X).fit()
        beta_mkt[col] = res.params['mkt']
        beta_divatt[col] = res.params['divatt']

    return start_date, beta_mkt, beta_divatt


results = Parallel(n_jobs=-1, backend='loky')(
    delayed(estimate_one_month)(i, month_ends, returns, MKT, DIVATT, rf_daily)
    for i in range(len(month_ends) - 1)
)

beta_mkt_df = pd.DataFrame(
    {start_date: beta_mkt for start_date, beta_mkt, beta_divatt in results}
).T.sort_index()

beta_divatt_df = pd.DataFrame(
    {start_date: beta_divatt for start_date, beta_mkt, beta_divatt in results}
).T.sort_index()

In [ ]:
beta_mkt_df.head()

## 4. Base Portfolio 구성

각 월말 $\tau$에서 종목을 먼저 시장 베타 기준으로 5분위로 나눈다.

$$
Q^{MKT}_{m,\tau}
= \left\{ i : \beta^{MKT}_{i,\tau} \in \text{MKT beta quintile } m \right\},
\quad m=1,\dots,5
$$

그 다음 각 시장 베타 분위 안에서 배당 관심도 베타 기준으로 다시 5분위로 나눈다.

$$
P_{m,n,\tau}
= \left\{ i \in Q^{MKT}_{m,\tau} :
\beta^{DIV}_{i,\tau} \in \text{DIV beta quintile } n \mid Q^{MKT}_{m,\tau} \right\},
\quad n=1,\dots,5
$$

따라서 총 25개의 base portfolio가 만들어진다.  
각 포트폴리오의 다음 달 수익률은 동일가중 평균으로 계산한다.

$$
R_{m,n,t+1}
= \frac{1}{N_{m,n,\tau}}
\sum_{i \in P_{m,n,\tau}} R_{i,t+1}
$$

이 25개 포트폴리오는 이후 2-step regression의 test asset으로 사용된다.


In [ ]:
returns_monthly = adj_close.resample("ME").last().pct_change(fill_method=None)

In [ ]:
n = 5   # 원하는 분위 수: 5면 5x5, 10이면 10x10

results = []

for i in range(len(month_ends) - 1):
    start_date = month_ends[i]
    end_date   = month_ends[i + 1]

    beta_mkt    = beta_mkt_df.loc[start_date].dropna()
    beta_divatt = beta_divatt_df.loc[start_date].dropna()
    ret_next    = returns_monthly.loc[end_date].dropna()

    common = beta_mkt.index.intersection(beta_divatt.index).intersection(ret_next.index)

    df = pd.DataFrame({
        'beta_mkt': beta_mkt.loc[common],
        'beta_divatt': beta_divatt.loc[common],
        'ret': ret_next.loc[common]
    })

    # 1차 정렬: beta_mkt n분위
    df['mkt_q'] = pd.qcut(df['beta_mkt'], n, labels=False, duplicates='drop') + 1

    # 2차 정렬: 각 mkt 분위 안에서 beta_divatt n분위
    df['div_q'] = (
        df.groupby('mkt_q')['beta_divatt']
          .transform(lambda x: pd.qcut(x, n, labels=False, duplicates='drop') + 1)
    )

    port_ret = df.groupby(['mkt_q', 'div_q'])['ret'].mean()

    row = {
        f'({m},{d})': port_ret.get((m, d), np.nan)
        for m in range(1, n + 1)
        for d in range(1, n + 1)
    }
    row['Date'] = end_date
    results.append(row)

portfolio_ret_nxn = pd.DataFrame(results).set_index('Date').sort_index()

In [ ]:
portfolio_ret_nxn.head()

In [ ]:
# 보고용: 5x5 base portfolio 평균수익률 표와 heatmap
import matplotlib.pyplot as plt
import seaborn as sns

mean_5x5 = portfolio_ret_nxn.mean().values.reshape(n, n)
mean_5x5_df = pd.DataFrame(
    mean_5x5,
    index=[f'MKT Q{i}' for i in range(1, n + 1)],
    columns=[f'DIV Q{i}' for i in range(1, n + 1)]
)

display(mean_5x5_df.style.format('{:.3%}'))

plt.figure(figsize=(7, 5))
sns.heatmap(mean_5x5_df, annot=True, fmt='.2%', cmap='RdYlGn', center=0)
plt.title('Average Monthly Returns of 5x5 Base Portfolios')
plt.xlabel('DIV beta quintile')
plt.ylabel('MKT beta quintile')
plt.tight_layout()
plt.show()


## 5. 2-step Regression

25개 base portfolio를 test asset으로 두고 2단계 회귀를 수행한다.

### Step 1. 포트폴리오별 factor loading 추정

각 포트폴리오 $p$에 대해 전체 표본 기간의 시계열 회귀를 수행한다.  
주의할 점은 배당 관심도 요인은 같은 달 값이 아니라 전월 값 $DIVATT_{t-1}$을 사용한다는 점이다.

$$
R^e_{p,t}
= a_p
+ b_{p,SMB}SMB_t
+ b_{p,HML}HML_t
+ b_{p,MOM}MOM_t
+ b_{p,MKT}MKT^e_t
+ b_{p,DIV}DIVATT_{t-1}
+ u_{p,t}
$$

즉 본 분석의 2-step 회귀식은 간단히 쓰면 다음과 같다.

$$
R_t = Factors_t + DIVATT_{t-1} + \varepsilon_t
$$

### Step 2. 월별 횡단면 회귀

Step 1에서 추정한 factor loading을 이용해 매월 포트폴리오 수익률을 설명한다.

$$
R^e_{p,t}
= \lambda_{0,t}
+ \lambda_{SMB,t}\hat{b}_{p,SMB}
+ \lambda_{HML,t}\hat{b}_{p,HML}
+ \lambda_{MOM,t}\hat{b}_{p,MOM}
+ \lambda_{MKT,t}\hat{b}_{p,MKT}
+ \lambda_{DIV,t}\hat{b}_{p,DIV}
+ \eta_{p,t}
$$

마지막으로 $\lambda_t$의 평균이 0과 다른지 Newey-West 표준오차로 검정한다.


In [ ]:
factors_monthly = factors.drop(columns=['RF']).resample("ME").last().pct_change(fill_method=None).loc[portfolio_ret_nxn.index]

In [ ]:
rf = factors['RF'].resample("ME").last()
rf_monthly = (1 + rf / 100) ** (1/12) - 1

factors_monthly['MKT']     = (MKT + 1).cumprod().resample("ME").last().pct_change().loc[portfolio_ret_nxn.index] - rf_monthly
# factors_monthly['DIVATT']  = np.log(SVI[keyword].resample("ME").last()).diff().loc[portfolio_ret_nxn.index]
factors_monthly['DIVATT']  = SVI[keyword].resample("ME").last().pct_change(fill_method=None).shift(1).loc[portfolio_ret_nxn.index]

In [ ]:
factors_monthly.tail()

In [ ]:
portfolio_excess_ret = portfolio_ret_nxn.sub(rf_monthly.reindex(portfolio_ret_nxn.index), axis=0)

In [ ]:
# Full sample OLS regression to get betas for each portfolio
X = sm.add_constant(factors_monthly)

betas = portfolio_excess_ret.apply(
    lambda y: sm.OLS(
        y.dropna(),
        X.loc[y.dropna().index]
    ).fit().params
).T

factor_betas = betas.drop(columns='const')

In [ ]:
# Fama-MacBeth regression to get lambda for each factor
X = sm.add_constant(factor_betas)

lambdas = pd.DataFrame(
    {
        date: sm.OLS(
            portfolio_excess_ret.loc[date].reindex(X.index),
            X
        ).fit().params.drop('const')
        for date in portfolio_excess_ret.index
    }
).T

In [ ]:
def nw_mean_test(x, lags=3):
    x = x.dropna()
    X = pd.DataFrame({'const': 1.0}, index=x.index)
    res = sm.OLS(x, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return pd.Series({
        'mean': res.params['const'],
        't_stat': res.tvalues['const'],
        'p_value': res.pvalues['const']
    })

fm_result = lambdas.apply(nw_mean_test, lags=3).T
fm_result

In [ ]:
# 보고용: Fama-MacBeth 위험가격 요약
fm_report = fm_result.copy()
fm_report['significance'] = pd.cut(
    fm_report['p_value'],
    bins=[-np.inf, 0.01, 0.05, 0.10, np.inf],
    labels=['***', '**', '*', '']
)

display(fm_report.style.format({
    'mean': '{:.4f}',
    't_stat': '{:.2f}',
    'p_value': '{:.4f}'
}))


## 6. 현재 결과 해석

원본 실행 시점(전체 표본, in-sample)의 결과에서 `DIVATT`의 평균 위험가격은 음수이며 통계적으로 유의하다.  
이는 배당 검색 관심도 변화에 더 민감한 포트폴리오가 이후 낮은 수익률을 보이는 경향과 일치한다.

경제적 해석은 다음과 같다.  
배당 관련 검색 관심이 증가할 때 투자자 수요가 특정 종목에 몰리면, 해당 종목이 단기적으로 과대평가되고 이후 기대수익률이 낮아질 수 있다.


## 7. DIV beta 5분위 알파 검증

추가로 종목을 $\beta^{DIV}$ 기준 5분위로만 나누어 월별 포트폴리오를 구성한다.  
각 분위 포트폴리오의 알파를 `DIVATT` 포함 모형과 미포함 모형에서 비교한다.

$$
R^e_{q,t}
= \alpha_q + \gamma_q'Factors_t + e_{q,t}
$$

`DIVATT`를 포함한 경우에는 요인 집합에 $DIVATT_{t-1}$이 추가된다.


In [ ]:
results = []

for i in range(len(month_ends) - 1):
    start_date = month_ends[i]
    end_date   = month_ends[i + 1]

    beta_divatt = beta_divatt_df.loc[start_date].dropna()
    ret_next    = returns_monthly.loc[end_date].dropna()

    common = beta_divatt.index.intersection(ret_next.index)

    df = pd.DataFrame({
        'beta_divatt': beta_divatt.loc[common],
        'ret': ret_next.loc[common]
    })

    df['div_q'] = pd.qcut(df['beta_divatt'], 5, labels=False, duplicates='drop') + 1

    port_ret = df.groupby('div_q')['ret'].mean()

    row = {f'Q{q}': port_ret.get(q, np.nan) for q in range(1, 6)}
    row['Date'] = end_date
    results.append(row)

portfolio_ret_div5 = pd.DataFrame(results).set_index('Date').sort_index()

# portfolio_ret_div5['Q5-Q1'] = portfolio_ret_div5['Q5'] - portfolio_ret_div5['Q1']

In [ ]:
portfolio_ret_div5.mean().plot(kind='bar', title='Average Monthly Returns by DIVATT Quintiles')
# (portfolio_ret_div5 + 1).cumprod().plot(figsize=(12, 8), title='Cumulative Returns by DIVATT Quintiles')

In [ ]:
def alpha_test(portfolio_ret, factors, use_divatt=True, lags=3):
    factor_cols = ['SMB', 'HML', 'MOM', 'MKT', 'DIVATT'] if use_divatt else ['SMB', 'HML', 'MOM', 'MKT']
    results = []

    for col in portfolio_ret.columns:
        df = pd.concat([
            portfolio_ret[col].rename('ret'),
            factors[factor_cols]
        ], axis=1).dropna()

        X = sm.add_constant(df[factor_cols])
        y = df['ret']

        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        results.append({
            'portfolio': col,
            'alpha': res.params['const'],
            'alpha_tstat': res.tvalues['const'],
            'alpha_pval': res.pvalues['const'],
            'R2': res.rsquared
        })

    return pd.DataFrame(results).set_index('portfolio')

alpha_with_divatt = alpha_test(portfolio_ret_div5, factors_monthly, use_divatt=True, lags=3)
alpha_without_divatt = alpha_test(portfolio_ret_div5, factors_monthly, use_divatt=False, lags=3)

In [ ]:
alpha_without_divatt

In [ ]:
alpha_with_divatt

In [ ]:
alpha_compare = pd.concat(
    [
        alpha_without_divatt.add_suffix('_without_DIVATT'),
        alpha_with_divatt.add_suffix('_with_DIVATT')
    ],
    axis=1
)

alpha_compare

In [ ]:
# portfolio_ret_nxn내에서z(n,m) 형태의 컬럼 이름 중 n이 1인 열만 선택
q1_portfolios = [col for col in portfolio_ret_nxn.columns if col.startswith('(5,')]
portfolio_ret_nxn[q1_portfolios].mean().mean()# .mean().plot(kind='bar', title='Average Returns of Portfolios with Market Beta in Q1')
q1_portfolios


In [ ]:
factor_betas.T[q1_portfolios].loc['DIVATT'].plot(kind='bar', title='DIVATT Betas for Portfolios with Market Beta in Q1')
factor_betas.T[q1_portfolios].loc['DIVATT']

---


In [ ]:
# 보고용: DIV beta 5분위 평균수익률과 알파 비교
avg_div5 = portfolio_ret_div5.mean().rename('avg_monthly_return').to_frame()
display(avg_div5.style.format('{:.3%}'))

alpha_report = alpha_compare[[
    'alpha_without_DIVATT', 'alpha_tstat_without_DIVATT',
    'alpha_with_DIVATT', 'alpha_tstat_with_DIVATT'
]]
display(alpha_report.style.format('{:.4f}'))

plt.figure(figsize=(6, 4))
portfolio_ret_div5.mean().plot(kind='bar')
plt.title('Average Monthly Returns by DIV beta Quintile')
plt.xlabel('DIV beta quintile')
plt.ylabel('Average monthly return')
plt.tight_layout()
plt.show()
